# 02. Limpieza y validación

Este notebook contiene las reglas, ejecución y validaciones del resultado final. El paso 0 queda reservado para la descarga automática y el notebook 01 cubre el diagnóstico previo.

In [15]:
"""Utilidades genericas de normalizacion de texto.

Estas funciones se usan sobre cualquier columna de texto libre. No corrigen
ortografia de nombres propios (eso requeriria criterio editorial que el
proyecto explicitamente prohibe aplicar de forma automatica); solo resuelven
problemas de forma: espacios, caracteres invisibles, comillas envolventes y
un typo comun de tildes.
"""

import re

# Espacio de no separacion, espacios de ancho variable y caracteres de ancho
# cero que a veces llegan de copiar/pegar desde Excel o Word.
_CARACTERES_INVISIBLES = re.compile(
    "[ ​‌‍⁠﻿\t]"
)
_ESPACIOS_MULTIPLES = re.compile(r" {2,}")
_PUNTUACION_SUELTA_BORDE_INICIO = re.compile(r"^[,;:\s]+")
_PUNTUACION_SUELTA_BORDE_FINAL = re.compile(r"[,;:\s]+$")

# El espanol no usa acento grave; cuando aparece es casi siempre un typo de
# teclado por el acento agudo correcto.
_ACENTOS_GRAVES = str.maketrans("àèìòùÀÈÌÒÙ", "áéíóúÁÉÍÓÚ")

_PLACEHOLDERS_FALTANTE = {
    "N/A", "NA", "N.A", "NULL", "NONE", "SIN DATO", "SIN DATOS", "S/D", "SD",
    "SIN REGISTRO",
}


def normalizar_espacios(valor: str) -> str:
    """Colapsa caracteres invisibles y espacios multiples; recorta bordes."""
    if valor is None:
        return ""
    limpio = _CARACTERES_INVISIBLES.sub(" ", valor)
    limpio = _ESPACIOS_MULTIPLES.sub(" ", limpio)
    return limpio.strip()


def corregir_acentos_graves(valor: str) -> str:
    """Sustituye acentos graves (ausentes en espanol) por su equivalente agudo."""
    return valor.translate(_ACENTOS_GRAVES)


def quitar_comillas_envolventes(valor: str) -> str:
    """Quita comillas dobles solo cuando envuelven la cadena COMPLETA
    (ej. '"LICEO LA SALLE"'). Si las comillas estan alrededor de una sola
    palabra dentro de un nombre mas largo (ej. COLEGIO "LA PATRIA"), se
    conservan porque son parte legitima del nombre del establecimiento."""
    if len(valor) >= 2 and valor.startswith('"') and valor.endswith('"'):
        interior = valor[1:-1]
        if '"' not in interior:
            return interior.strip()
    return valor


def quitar_puntuacion_suelta(valor: str) -> str:
    """Quita comas/puntoycoma/dos puntos sueltos al inicio o final (ej.
    'CATALINA PERALTA CARRILLO,' o '10A. AVENIDA 2-24,'). No toca el punto
    final: en DIRECCION casi siempre es parte de una abreviatura (ej. 'ZONA 4.')."""
    limpio = _PUNTUACION_SUELTA_BORDE_INICIO.sub("", valor)
    limpio = _PUNTUACION_SUELTA_BORDE_FINAL.sub("", limpio)
    return limpio


def limpiar_texto_general(valor: str) -> str:
    """Pipeline estandar para columnas de texto libre: espacios -> tildes
    invalidas -> comillas envolventes -> puntuacion suelta en los bordes."""
    limpio = normalizar_espacios(valor)
    limpio = corregir_acentos_graves(limpio)
    limpio = quitar_comillas_envolventes(limpio)
    limpio = quitar_puntuacion_suelta(limpio)
    return limpio


def es_placeholder_faltante(valor: str) -> bool:
    """Detecta valores que representan 'sin dato' aunque no esten vacios:
    solo puntuacion/guiones repetidos (---, ., ..), solo X/0 repetidos, o
    literales conocidos (N/A, NULL, SIN DATO, S/D)."""
    limpio = valor.strip()
    if limpio == "":
        return True
    if re.fullmatch(r"[-.,_]+", limpio):
        return True
    if re.fullmatch(r"[xX0]+", limpio):
        return True
    if limpio.upper() in _PLACEHOLDERS_FALTANTE:
        return True
    return False


In [16]:
"""Catalogo oficial de departamentos y municipios de Guatemala.

Fuente: INE/Wikipedia (https://es.wikipedia.org/wiki/Anexo:Municipios_de_Guatemala),
verificado contra los datos del proyecto. "CIUDAD CAPITAL" no es un
departamento oficial: es una particion propia de MINEDUC sobre el municipio
de Guatemala, reportada por zona (ZONA 1..25) en vez de por municipio.

3 valores de MUNICIPIO se dejan deliberadamente fuera del catalogo por no
calzar con ninguna forma oficial (QUEZALTEPEQUE, PACHALUN, SAN MIGUEL PANAM):
ver AvanceProyecto1.md para el detalle y el razonamiento completo.
"""

from unidecode import unidecode

DEPARTAMENTOS_MUNICIPIOS: dict[str, list[str]] = {
    "Alta Verapaz": [
        "Chahal", "Chisec", "Cobán", "Fray Bartolomé de las Casas",
        "Santa Catalina La Tinta", "Lanquín", "Panzós", "Raxruhá",
        "San Cristóbal Verapaz", "San Juan Chamelco", "San Pedro Carchá",
        "Santa Cruz Verapaz", "Cahabón", "Santa María Cahabón", "Senahú",
        "Tamahú", "Tactic", "Tucurú", "San Miguel Tucurú", "La Tinta",
    ],
    "Baja Verapaz": [
        "Cubulco", "Granados", "Purulhá", "Rabinal", "Salamá", "San Jerónimo",
        "San Miguel Chicaj", "Santa Cruz el Chol",
    ],
    "Chimaltenango": [
        "Acatenango", "Chimaltenango", "El Tejar", "Parramos", "Patzicía",
        "Patzún", "Pochuta", "San Miguel Pochuta", "San Andrés Itzapa",
        "San José Poaquíl", "San Juan Comalapa", "San Martín Jilotepeque",
        "Santa Apolonia", "Santa Cruz Balanyá", "Tecpán", "Tecpán Guatemala",
        "Yepocapa", "San Pedro Yepocapa", "Zaragoza",
    ],
    "Chiquimula": [
        "Camotán", "Chiquimula", "Concepción Las Minas", "Esquipulas", "Ipala",
        "Jocotán", "Olopa", "Quetzaltepeque", "San Jacinto", "San José la Arada",
        "San Juan Ermita",
    ],
    "El Progreso": [
        "El Jícaro", "Guastatoya", "Morazán", "San Agustín Acasaguastlán",
        "San Antonio La Paz", "San Cristóbal Acasaguastlán", "Sanarate", "Sansare",
    ],
    "Escuintla": [
        "Escuintla", "Guanagazapa", "Iztapa", "La Democracia", "La Gomera",
        "Masagua", "Nueva Concepción", "Palín", "San José", "San Vicente Pacaya",
        "Santa Lucía Cotzumalguapa", "Sipacate", "Siquinalá", "Tiquisate",
    ],
    "Guatemala": [
        "Amatitlán", "Chinautla", "Chuarrancho", "Guatemala", "Fraijanes",
        "Mixco", "Palencia", "San José del Golfo", "San José Pinula",
        "San Juan Sacatepéquez", "San Miguel Petapa", "San Pedro Ayampuc",
        "San Pedro Sacatepéquez", "San Raymundo", "Santa Catarina Pinula",
        "Villa Canales", "Villa Nueva",
    ],
    "Huehuetenango": [
        "Aguacatán", "Chiantla", "Colotenango", "Concepción Huista", "Cuilco",
        "Huehuetenango", "Jacaltenango", "La Democracia", "La Libertad",
        "Malacatancito", "Nentón", "Petatán", "San Antonio Huista",
        "San Gaspar Ixchil", "San Ildefonso Ixtahuacán", "San Juan Atitán",
        "San Juan Ixcoy", "San Mateo Ixtatán", "San Miguel Acatán",
        "San Pedro Nectá", "San Pedro Soloma", "San Rafael La Independencia",
        "San Rafael Pétzal", "San Sebastián Coatán", "San Sebastián Huehuetenango",
        "Santa Ana Huista", "Santa Bárbara", "Santa Cruz Barillas",
        "Santa Eulalia", "Santiago Chimaltenango", "Tectitán",
        "Todos Santos Cuchumatán", "Unión Cantinil",
    ],
    "Izabal": ["El Estor", "Livingston", "Los Amates", "Morales", "Puerto Barrios"],
    "Jalapa": [
        "Jalapa", "Mataquescuintla", "Monjas", "San Carlos Alzatate",
        "San Luis Jilotepeque", "San Manuel Chaparrón", "San Pedro Pinula",
    ],
    "Jutiapa": [
        "Agua Blanca", "Asunción Mita", "Atescatempa", "Comapa", "Conguaco",
        "El Adelanto", "El Progreso", "Jalpatagua", "Jerez", "Jutiapa", "Moyuta",
        "Pasaco", "Quesada", "San José Acatempa", "Santa Catarina Mita",
        "Yupiltepeque", "Zapotitlán",
    ],
    "Petén": [
        "Dolores", "El Chal", "Flores", "Santa Elena de la Cruz", "La Libertad",
        "Las Cruces", "Melchor de Mencos", "Poptún", "San Andrés", "San Benito",
        "San Francisco", "San José", "San Luis", "Santa Ana", "Sayaxché",
    ],
    "Quetzaltenango": [
        "Almolonga", "Cabricán", "Cajolá", "Cantel", "Coatepeque",
        "Colomba Costa Cuca", "Concepción Chiquirichapa", "El Palmar",
        "Flores Costa Cuca", "Génova", "Génova Costa Cuca", "Huitán",
        "La Esperanza", "Olintepeque",
        "Palestina de Los Altos", "Quetzaltenango", "Salcajá", "San Carlos Sija",
        "San Francisco La Unión", "San Juan Ostuncalco", "San Martín Sacatepéquez",
        "San Mateo", "San Miguel Sigüilá", "Sibilia", "Zunil",
    ],
    "Quiché": [
        "Canillá", "Chajul", "Chicamán", "Chiché", "Santo Tomás Chichicastenango",
        "Chinique", "Cunén", "Ixcán", "Joyabaj", "Nebaj", "Pachalum", "Patzité",
        "Sacapulas", "San Andrés Sajcabajá", "San Antonio Ilotenango",
        "San Bartolomé Jocotenango", "San Juan Cotzal", "San Pedro Jocopilas",
        "Santa Cruz del Quiché", "Uspantán", "San Miguel Uspantán", "Zacualpa",
    ],
    "Retalhuleu": [
        "Champerico", "El Asintal", "Nuevo San Carlos", "Retalhuleu",
        "San Andrés Villa Seca", "San Felipe", "San Martín Zapotitlán",
        "San Sebastián", "Santa Cruz Muluá",
    ],
    "Sacatepéquez": [
        "Alotenango", "Ciudad Vieja", "Jocotenango", "Antigua Guatemala",
        "Magdalena Milpas Altas", "Pastores", "San Antonio Aguas Calientes",
        "San Bartolomé Milpas Altas", "San Lucas Sacatepéquez",
        "San Miguel Dueñas", "Santa Catarina Barahona", "Santa Lucía Milpas Altas",
        "Santa María de Jesús", "Santiago Sacatepéquez", "Santo Domingo Xenacoj",
        "Sumpango",
    ],
    "San Marcos": [
        "Ayutla", "Catarina", "Comitancillo", "Concepción Tutuapa", "El Quetzal",
        "El Tumbador", "Esquipulas Palo Gordo", "Ixchiguán", "La Blanca",
        "La Reforma", "Malacatán", "Nuevo Progreso", "Ocós", "Pajapita",
        "Río Blanco", "San Antonio Sacatepéquez", "San Cristóbal Cucho",
        "San José El Rodeo", "San José Ojetenam", "San Lorenzo", "San Marcos",
        "San Miguel Ixtahuacán", "San Pablo", "San Pedro Sacatepéquez",
        "San Rafael Pie de la Cuesta", "Sibinal", "Sipacapa", "Tacaná",
        "Tajumulco", "Tejutla",
    ],
    "Santa Rosa": [
        "Barberena", "Casillas", "Chiquimulilla", "Cuilapa", "Guazacapán",
        "Nueva Santa Rosa", "Oratorio", "Pueblo Nuevo Viñas", "San Juan Tecuaco",
        "San Rafael las Flores", "Santa Cruz Naranjo", "Santa María Ixhuatán",
        "Santa Rosa de Lima", "Taxisco",
    ],
    "Sololá": [
        "Concepción", "Nahualá", "Panajachel", "San Andrés Semetabaj",
        "San Antonio Palopó", "San José Chacayá", "San Juan La Laguna",
        "San Lucas Tolimán", "San Marcos La Laguna", "San Pablo La Laguna",
        "San Pedro La Laguna", "Santa Catarina Ixtahuacán",
        "Santa Catarina Palopó", "Santa Clara La Laguna", "Santa Cruz La Laguna",
        "Santa Lucía Utatlán", "Santa María Visitación", "Santiago Atitlán",
        "Sololá",
    ],
    "Suchitepéquez": [
        "Chicacao", "Cuyotenango", "Mazatenango", "Patulul", "Pueblo Nuevo",
        "Río Bravo", "Samayac", "San Antonio Suchitepéquez", "San Bernardino",
        "San Francisco Zapotitlán", "San Gabriel", "San José El Ídolo",
        "San José La Máquina", "San Juan Bautista", "San Lorenzo",
        "San Miguel Panán", "San Pablo Jocopilas", "Santa Bárbara",
        "Santo Domingo Suchitepéquez", "Santo Tomás La Unión", "Zunilito",
    ],
    "Totonicapán": [
        "Momostenango", "San Andrés Xecul", "San Bartolo Aguas Calientes",
        "San Cristóbal Totonicapán", "San Francisco El Alto",
        "Santa Lucía La Reforma", "Santa María Chiquimula", "Totonicapán",
    ],
    "Zacapa": [
        "Cabañas", "Estanzuela", "Gualán", "Huité", "La Unión", "Río Hondo",
        "San Diego", "San Jorge", "Teculután", "Usumatlán", "Zacapa",
    ],
}

# Nombres oficiales con tilde para presentacion (clave = version sin tilde en
# mayusculas, tal como se usa para las comparaciones internas).
DEPARTAMENTOS_CANONICOS = {
    "PETEN": "PETÉN",
    "QUICHE": "QUICHÉ",
    "SACATEPEQUEZ": "SACATEPÉQUEZ",
    "SOLOLA": "SOLOLÁ",
    "SUCHITEPEQUEZ": "SUCHITEPÉQUEZ",
    "TOTONICAPAN": "TOTONICAPÁN",
}

# "Ciudad Capital" no es un departamento oficial de Guatemala: es una
# particion administrativa propia de MINEDUC sobre el municipio de Guatemala,
# reportada por zona en vez de por municipio.
DEPARTAMENTO_ESPECIAL_CIUDAD_CAPITAL = "CIUDAD CAPITAL"
ZONAS_CIUDAD_CAPITAL_VALIDAS = {f"ZONA {i}" for i in range(1, 26)}


def _clave(texto: str) -> str:
    return unidecode(texto).strip().upper()


def departamento_canonico(valor: str) -> str:
    clave = _clave(valor)
    if clave == DEPARTAMENTO_ESPECIAL_CIUDAD_CAPITAL:
        return DEPARTAMENTO_ESPECIAL_CIUDAD_CAPITAL
    return DEPARTAMENTOS_CANONICOS.get(clave, clave)


def departamento_valido(valor: str) -> bool:
    clave = _clave(valor)
    if clave == DEPARTAMENTO_ESPECIAL_CIUDAD_CAPITAL:
        return True
    return clave in {_clave(d) for d in DEPARTAMENTOS_MUNICIPIOS}


_MUNICIPIOS_POR_DEPTO_CLAVE: dict[str, dict[str, str]] = {
    _clave(depto): {_clave(m): m.upper() for m in municipios}
    for depto, municipios in DEPARTAMENTOS_MUNICIPIOS.items()
}


def municipio_canonico(departamento: str, municipio: str) -> str:
    """Devuelve el nombre de municipio normalizado (mayusculas, con tilde
    cuando el catalogo oficial la tiene). Si no se reconoce, regresa el valor
    de entrada normalizado en mayusculas sin alterar su contenido."""
    depto_clave = _clave(departamento)
    if depto_clave == DEPARTAMENTO_ESPECIAL_CIUDAD_CAPITAL:
        return municipio.strip().upper()
    catalogo = _MUNICIPIOS_POR_DEPTO_CLAVE.get(depto_clave, {})
    return catalogo.get(_clave(municipio), municipio.strip().upper())


def municipio_valido(departamento: str, municipio: str) -> bool:
    depto_clave = _clave(departamento)
    muni_clave = _clave(municipio)
    if depto_clave == DEPARTAMENTO_ESPECIAL_CIUDAD_CAPITAL:
        return municipio.strip().upper() in ZONAS_CIUDAD_CAPITAL_VALIDAS
    catalogo = _MUNICIPIOS_POR_DEPTO_CLAVE.get(depto_clave)
    if catalogo is None:
        return False
    return muni_clave in catalogo


In [17]:
"""Limpieza especifica por columna (excepto TELEFONO, en ``phones.py``)."""

import re

import pandas as pd
from unidecode import unidecode


PATRON_CODIGO = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
PATRON_DISTRITO_DETALLADO = re.compile(r"^\d{2}-\d{2}-\d{4}$")
PATRON_DISTRITO_CORTO = re.compile(r"^\d{2}-\d{3}$")
PATRON_DISTRITO_INCOMPLETO = re.compile(r"^\d{2}-$")


def procesar_codigo(serie: pd.Series) -> tuple[pd.Series, pd.Series]:
    """CODIGO ya cumple el patron ##-##-####-## en el 100% de los registros
    del crudo; se conserva como texto y se devuelve una bandera de validez
    para dejar la verificacion explicita en vez de asumirla."""
    limpio = serie.map(normalizar_espacios)
    valido = limpio.str.match(PATRON_CODIGO)
    return limpio, valido


def clasificar_distrito(valor: str) -> str:
    if valor == "":
        return "vacio"
    if PATRON_DISTRITO_DETALLADO.match(valor):
        return "detallado"
    if PATRON_DISTRITO_CORTO.match(valor):
        return "corto"
    if PATRON_DISTRITO_INCOMPLETO.match(valor):
        return "incompleto"
    return "formato_no_reconocido"


def procesar_distrito(serie: pd.Series) -> tuple[pd.Series, pd.Series, pd.Series]:
    """DISTRITO conviven dos esquemas validos de MINEDUC (##-## y
    ##-##-####); se documentan ambos en vez de forzar uno solo. Los valores
    truncados (##-) o vacios se marcan como invalidos, no se completan."""
    limpio = serie.map(normalizar_espacios)
    clase = limpio.map(clasificar_distrito)
    valido = clase.isin(["detallado", "corto"])
    # Solo se marca NA lo realmente vacio; "incompleto" (ej. "01-") conserva
    # el prefijo de departamento que si trae, y se senala via DISTRITO_VALIDO.
    limpio = limpio.mask(clase == "vacio", pd.NA)
    return limpio, clase, valido


def procesar_departamento(serie: pd.Series) -> tuple[pd.Series, pd.Series]:
    limpio = serie.map(normalizar_espacios).map(Catalogos.departamento_canonico)
    valido = serie.map(Catalogos.departamento_valido)
    return limpio, valido


def procesar_municipio(departamento: pd.Series, municipio: pd.Series) -> tuple[pd.Series, pd.Series]:
    municipio_limpio = municipio.map(normalizar_espacios)
    limpio = [
        Catalogos.municipio_canonico(d, m) for d, m in zip(departamento, municipio_limpio)
    ]
    valido = [
        Catalogos.municipio_valido(d, m) for d, m in zip(departamento, municipio_limpio)
    ]
    return pd.Series(limpio, index=municipio.index), pd.Series(valido, index=municipio.index)


def procesar_categorica_simple(serie: pd.Series) -> pd.Series:
    """Para columnas de dominio pequeno que el perfilado ya mostro limpias
    (NIVEL, SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN, DEPARTAMENTAL):
    solo normaliza espacios/mayusculas; no se detectaron variantes de
    escritura para unificar en el crudo."""
    return serie.map(normalizar_espacios).str.upper()


def _variante_mas_correcta(variantes: list[str]) -> str:
    """Entre variantes que solo difieren en tildes (misma palabra sin
    acentos), prefiere la forma con mas vocales acentuadas: el espanol
    requiere la tilde, quitarla es el error, no al reves."""
    return max(variantes, key=lambda v: (sum(1 for c in v if c in "ÁÉÍÓÚ"), v))


def unificar_variantes_por_tilde(serie: pd.Series) -> tuple[pd.Series, dict[str, str]]:
    """Unifica registros que son identicos salvo por tildes (incluye typos
    de acento grave ya corregidos antes de esta funcion): por ejemplo
    'CARLOS HUMBERTO GONZALEZ DE LEON' y 'CARLOS HUMBERTO GONZÁLEZ DE LEÓN'
    en SUPERVISOR, o variantes equivalentes en ESTABLECIMIENTO y DIRECTOR.
    No toca diferencias de letras, orden de palabras o abreviaturas
    distintas: esas quedan para el analisis de duplicados parciales, donde
    cada caso se documenta uno por uno en vez de fusionarse automaticamente.
    Ignora los valores NA (no hay nada que unificar en un dato faltante)."""
    no_nulos = serie.dropna()
    base = no_nulos.map(unidecode)
    grupos = no_nulos.groupby(base).unique()
    mapa: dict[str, str] = {}
    for variantes in grupos:
        if len(variantes) > 1:
            canonico = _variante_mas_correcta(list(variantes))
            for variante in variantes:
                if variante != canonico:
                    mapa[variante] = canonico
    limpio = serie.replace(mapa)
    return limpio, mapa


def procesar_texto_libre_con_placeholder(serie: pd.Series) -> tuple[pd.Series, dict[str, str]]:
    """Para DIRECTOR, SUPERVISOR y DIRECCION: normaliza texto, unifica
    variantes que solo difieren en tildes, y convierte placeholders (---,
    ..., S/D, SIN REGISTRO, vacios) en NA real para que se cuenten como
    faltantes en vez de como texto valido."""
    limpio = serie.map(limpiar_texto_general)
    limpio, mapa_tildes = unificar_variantes_por_tilde(limpio)
    limpio = limpio.mask(limpio.map(es_placeholder_faltante), pd.NA)
    return limpio, mapa_tildes


def procesar_establecimiento(serie: pd.Series) -> tuple[pd.Series, dict[str, str]]:
    limpio = serie.map(limpiar_texto_general)
    limpio, mapa_tildes = unificar_variantes_por_tilde(limpio)
    # No se puede inventar el nombre de un establecimiento: los vacios se
    # marcan NA explicito en vez de quedar como cadena vacia silenciosa.
    limpio = limpio.mask(limpio == "", pd.NA)
    return limpio, mapa_tildes


In [18]:
"""Normalizacion de la columna TELEFONO.

El guion es ambiguo (formato interno de un numero, separador entre varios
numeros, o sufijo abreviado que comparte prefijo con el numero anterior,
ej. "22202870-73" = 22202870 y 22202873) y FAX/EXT/" AL " marcan casos que
no se deben leer como una linea normal. Detalle completo y riesgos en
AvanceProyecto1.md. El valor crudo se conserva en TELEFONO_ORIGINAL para
poder auditar cualquier numero reconstruido.
"""

import re
from dataclasses import dataclass, field

import pandas as pd


_MARCA_FAX = re.compile(r"\bFAX\b", re.IGNORECASE)
_MARCA_EXT = re.compile(r"\bEST[XA]?\.?\b|\bEXT\.?\b", re.IGNORECASE)
_MARCA_RANGO = re.compile(r"\bAL\b", re.IGNORECASE)
_SEPARADOR_COARSE = re.compile(r"\s*(?:,|/|\bY\b)\s*", re.IGNORECASE)


@dataclass
class TelefonoParseado:
    numeros: list[str] = field(default_factory=list)
    tiene_fax: bool = False
    tiene_extension: bool = False
    tiene_rango_no_expandido: bool = False
    longitudes_invalidas: list[str] = field(default_factory=list)


def _dividir_token_con_guiones(token: str) -> list[str]:
    partes = token.split("-")
    partes = [p for p in partes if p != ""]
    if len(partes) <= 1:
        return partes

    largas = [p for p in partes if len(p) >= 6]
    if len(largas) == len(partes):
        # Todas las partes ya son numeros completos (8-8, 7-7, 8-7, etc.)
        return partes

    # Primer trozo es el numero de referencia; los siguientes son sufijos
    # cortos que comparten su prefijo, o un formato interno tipo XXX-XXXX.
    primero, resto = partes[0], partes[1:]
    if len(resto) == 1 and len(primero) + len(resto[0]) in (7, 8) and len(primero) in (3, 4):
        # Formato interno de un solo numero, ej. "999-9999" o "9999-9999"
        return [primero + resto[0]]

    resultado = [primero]
    for parte in resto:
        if len(parte) >= 6:
            resultado.append(parte)
        else:
            prefijo = primero[: len(primero) - len(parte)]
            resultado.append(prefijo + parte)
    return resultado


def parsear_telefono(valor: str) -> TelefonoParseado:
    resultado = TelefonoParseado()
    texto = normalizar_espacios(valor)
    if texto == "":
        return resultado

    if _MARCA_RANGO.search(texto):
        resultado.tiene_rango_no_expandido = True
        return resultado

    if _MARCA_FAX.search(texto):
        resultado.tiene_fax = True
        texto = _MARCA_FAX.split(texto)[0]
    if _MARCA_EXT.search(texto):
        resultado.tiene_extension = True
        texto = _MARCA_EXT.split(texto)[0]

    texto = normalizar_espacios(texto)
    if texto == "":
        return resultado

    tokens_gruesos = [t for t in _SEPARADOR_COARSE.split(texto) if t.strip()]
    numeros: list[str] = []
    for token in tokens_gruesos:
        token = token.strip()
        if not re.fullmatch(r"[\d\- ]+", token):
            # Quedo texto no reconocido (abreviaturas no previstas); se marca
            # como longitud invalida en vez de intentar adivinar su forma.
            resultado.longitudes_invalidas.append(token)
            continue
        for sub in token.split():
            for numero in _dividir_token_con_guiones(sub):
                digitos = re.sub(r"\D", "", numero)
                if digitos == "":
                    continue
                if len(digitos) in (7, 8):
                    numeros.append(digitos)
                else:
                    resultado.longitudes_invalidas.append(digitos)

    resultado.numeros = numeros
    return resultado


def formato_valido(parseado: TelefonoParseado) -> bool:
    if not parseado.numeros:
        return False
    if parseado.longitudes_invalidas or parseado.tiene_rango_no_expandido:
        return False
    return all(len(n) == 8 for n in parseado.numeros)


def observacion(parseado: TelefonoParseado, original: str) -> str:
    if normalizar_espacios(original) == "":
        return "vacio"
    notas = []
    if parseado.tiene_rango_no_expandido:
        notas.append("rango no expandido (revision manual)")
    if parseado.tiene_fax:
        notas.append("incluye FAX")
    if parseado.tiene_extension:
        notas.append("incluye extension")
    if parseado.longitudes_invalidas:
        notas.append(f"longitud invalida: {', '.join(parseado.longitudes_invalidas)}")
    if any(len(n) == 7 for n in parseado.numeros):
        notas.append("formato antiguo de 7 digitos")
    if len(parseado.numeros) > 1:
        notas.append(f"{len(parseado.numeros)} telefonos en una celda")
    return "; ".join(notas) if notas else "ok"


def procesar_telefono(valores: list[str]) -> dict[str, list]:
    limpio, cantidad, valido, obs = [], [], [], []
    for original in valores:
        parseado = parsear_telefono(original)
        limpio.append("; ".join(parseado.numeros) if parseado.numeros else pd.NA)
        cantidad.append(len(parseado.numeros))
        valido.append(formato_valido(parseado))
        obs.append(observacion(parseado, original))
    return {
        "TELEFONO": limpio,
        "TELEFONO_CANTIDAD": cantidad,
        "TELEFONO_VALIDO": valido,
        "TELEFONO_OBSERVACION": obs,
    }


In [19]:
"""Deteccion de duplicados exactos y parciales.

Los duplicados parciales solo se listan para revision manual (no se
fusionan/eliminan automaticamente, por pedido explicito de la guia). Se usa
"blocking" por DEPARTAMENTO+MUNICIPIO para que comparar 11,867 registros sea
viable.
"""

import pandas as pd
from rapidfuzz import fuzz, process


def detectar_duplicados_exactos(df: pd.DataFrame) -> pd.Series:
    return df.duplicated(keep=False)


def detectar_duplicados_parciales(
    df: pd.DataFrame,
    columna_nombre: str = "ESTABLECIMIENTO",
    columna_bloque: tuple[str, str] = ("DEPARTAMENTO", "MUNICIPIO"),
    umbral: float = 90.0,
) -> pd.DataFrame:
    pares = []
    col_dep, col_mun = columna_bloque
    for (_, _), grupo in df.groupby([col_dep, col_mun]):
        if len(grupo) < 2:
            continue
        nombres = grupo[columna_nombre].tolist()
        indices = grupo.index.tolist()
        matriz = process.cdist(nombres, nombres, scorer=fuzz.token_sort_ratio)
        n = len(nombres)
        for i in range(n):
            for j in range(i + 1, n):
                similitud = matriz[i, j]
                if similitud >= umbral:
                    fila_a, fila_b = grupo.iloc[i], grupo.iloc[j]
                    pares.append(
                        {
                            "indice_a": indices[i],
                            "indice_b": indices[j],
                            "codigo_a": fila_a["CODIGO"],
                            "codigo_b": fila_b["CODIGO"],
                            "establecimiento_a": fila_a[columna_nombre],
                            "establecimiento_b": fila_b[columna_nombre],
                            "similitud_nombre": round(float(similitud), 1),
                            "misma_direccion": fila_a.get("DIRECCION") == fila_b.get("DIRECCION"),
                            "mismo_telefono": fila_a.get("TELEFONO") == fila_b.get("TELEFONO")
                            and fila_a.get("TELEFONO", "") != "",
                            "status_a": fila_a.get("STATUS"),
                            "status_b": fila_b.get("STATUS"),
                            "decision": "pendiente de revision manual",
                        }
                    )
    columnas = [
        "indice_a", "indice_b", "codigo_a", "codigo_b",
        "establecimiento_a", "establecimiento_b", "similitud_nombre",
        "misma_direccion", "mismo_telefono", "status_a", "status_b", "decision",
    ]
    resultado = pd.DataFrame(pares, columns=columnas)
    return resultado.sort_values("similitud_nombre", ascending=False).reset_index(drop=True)


def resumir_nombres_repetidos(
    df: pd.DataFrame,
    columna_nombre: str = "ESTABLECIMIENTO",
    columna_bloque: tuple[str, str] = ("DEPARTAMENTO", "MUNICIPIO"),
) -> pd.DataFrame:
    """Cuando el mismo nombre de establecimiento aparece varias veces en el
    mismo municipio, casi siempre es porque MINEDUC registra un renglon por
    cada combinacion de jornada/plan/sector (o por cambios de STATUS a lo
    largo del tiempo), no porque el dato este duplicado por error. Esta
    tabla resume esos grupos para que la revision manual se enfoque en los
    casos con distinta direccion (mas sospechosos) y no en los miles de
    filas con nombre 100% identico que ya tienen explicacion."""
    col_dep, col_mun = columna_bloque
    filas = []
    for (departamento, municipio, nombre), grupo in df.groupby([col_dep, col_mun, columna_nombre]):
        if len(grupo) < 2:
            continue
        filas.append(
            {
                "departamento": departamento,
                "municipio": municipio,
                "establecimiento": nombre,
                "registros": len(grupo),
                "direcciones_distintas": grupo["DIRECCION"].nunique(),
                "status_distintos": grupo["STATUS"].nunique(),
                "jornadas_distintas": grupo["JORNADA"].nunique(),
                "planes_distintos": grupo["PLAN"].nunique(),
                "codigos": "; ".join(grupo["CODIGO"]),
            }
        )
    resultado = pd.DataFrame(
        filas,
        columns=[
            "departamento", "municipio", "establecimiento", "registros",
            "direcciones_distintas", "status_distintos", "jornadas_distintas",
            "planes_distintos", "codigos",
        ],
    )
    return resultado.sort_values(
        ["direcciones_distintas", "registros"], ascending=[False, False]
    ).reset_index(drop=True)


In [20]:
from types import SimpleNamespace
from pathlib import Path


def raiz():
    return next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").exists())


Catalogos = SimpleNamespace(
    departamento_canonico=departamento_canonico,
    departamento_valido=departamento_valido,
    municipio_canonico=municipio_canonico,
    municipio_valido=municipio_valido,
)
Columnas = SimpleNamespace(
    procesar_codigo=procesar_codigo,
    procesar_distrito=procesar_distrito,
    procesar_departamento=procesar_departamento,
    procesar_municipio=procesar_municipio,
    procesar_establecimiento=procesar_establecimiento,
    procesar_texto_libre_con_placeholder=procesar_texto_libre_con_placeholder,
    procesar_categorica_simple=procesar_categorica_simple,
)
Duplicados = SimpleNamespace(
    detectar_duplicados_parciales=detectar_duplicados_parciales,
    resumir_nombres_repetidos=resumir_nombres_repetidos,
)

In [21]:
"""Limpieza reproducible del conjunto consolidado de establecimientos educativos.

Lee `Data/csv/EstablecimientosDiversificadoConsolidado.csv` (salida de
`establecimientos.helpers.xls`), aplica las reglas de limpieza y exporta:

- Data/csv/EstablecimientosDiversificadoLimpio.csv   (dataset limpio)
- Reportes/RegistroTransformaciones.csv               (item 6 de la guia)
- Reportes/InformeCalidadAntesDespues.csv              (item 8 de la guia)
- Reportes/PosiblesDuplicadosParciales.csv             (item 5.g de la guia)

El notebook 02 invoca ``main``; este módulo no es una etapa independiente.
"""

from pathlib import Path

import pandas as pd


ROOT = raiz()
ENTRADA = ROOT / "Data" / "csv" / "EstablecimientosDiversificadoConsolidado.csv"
SALIDA_CSV = ROOT / "Data" / "csv" / "EstablecimientosDiversificadoLimpio.csv"
REPORTES_DIR = ROOT / "Reportes"

COLUMNAS_ORIGINALES = [
    "CODIGO", "DISTRITO", "DEPARTAMENTO", "MUNICIPIO", "ESTABLECIMIENTO",
    "DIRECCION", "TELEFONO", "SUPERVISOR", "DIRECTOR", "NIVEL", "SECTOR",
    "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL",
]
COLUMNAS_CATEGORICAS_SIMPLES = [
    "NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL",
]

registro_transformaciones: list[dict] = []


def registrar(variable: str, problema: str, transformacion: str, registros_afectados: int, justificacion: str) -> None:
    registro_transformaciones.append(
        {
            "Variable": variable,
            "Problema detectado": problema,
            "Transformacion": transformacion,
            "Registros afectados": registros_afectados,
            "Justificacion": justificacion,
        }
    )


def cargar_crudo() -> pd.DataFrame:
    return pd.read_csv(ENTRADA, dtype=str, keep_default_na=False, encoding="utf-8")


def contar_faltantes_estilo_crudo(df: pd.DataFrame) -> tuple[int, int]:
    faltantes = 0
    variables_con_na = 0
    for col in COLUMNAS_ORIGINALES:
        n = int(df[col].map(es_placeholder_faltante).sum())
        faltantes += n
        variables_con_na += int(n > 0)
    return faltantes, variables_con_na


def snapshot_calidad_despues(df: pd.DataFrame) -> dict:
    columnas_dato = [c for c in COLUMNAS_ORIGINALES if c in df.columns]
    faltantes = int(df[columnas_dato].isna().sum().sum())
    variables_con_na = int((df[columnas_dato].isna().sum() > 0).sum())
    total_celdas = len(df) * len(columnas_dato)
    return {
        "Registros": len(df),
        "Variables": len(df.columns),
        "Valores faltantes": faltantes,
        "% faltantes": round(100 * faltantes / total_celdas, 2),
        "Variables con NA": variables_con_na,
        "Duplicados exactos": int(df[COLUMNAS_ORIGINALES].duplicated(keep=False).sum()),
    }


def limpiar(df_crudo: pd.DataFrame) -> pd.DataFrame:
    df = df_crudo.copy()

    # --- CODIGO ---------------------------------------------------------
    limpio, valido = Columnas.procesar_codigo(df["CODIGO"])
    df["CODIGO"] = limpio
    registrar(
        "CODIGO", "Ninguno: el 100% de los registros ya cumple ##-##-####-##.",
        "Se conserva como texto y se agrega verificacion de patron.",
        int((~valido).sum()),
        "Es un identificador, no una cantidad; convertirlo a numero perderia los guiones y ceros a la izquierda.",
    )

    # --- DISTRITO ---------------------------------------------------------
    limpio, clase, valido = Columnas.procesar_distrito(df["DISTRITO"])
    df["DISTRITO"] = limpio
    n_incompleto = int((clase == "incompleto").sum())
    n_vacio = int((clase == "vacio").sum())
    registrar(
        "DISTRITO",
        f"{n_vacio} vacios y {n_incompleto} truncados (ej. '01-' sin numero de distrito).",
        "Se documentan dos esquemas validos (##-### y ##-##-####) y se marca DISTRITO_VALIDO=False en vacios/truncados.",
        n_vacio + n_incompleto,
        "No se puede inventar el numero de distrito faltante; se deja explicito para seguimiento con la fuente.",
    )
    df["DISTRITO_FORMATO"] = clase
    df["DISTRITO_VALIDO"] = valido

    # --- DEPARTAMENTO -----------------------------------------------------
    limpio, valido = Columnas.procesar_departamento(df["DEPARTAMENTO"])
    n_tilde = int((limpio != df["DEPARTAMENTO"]).sum())
    df["DEPARTAMENTO"] = limpio
    registrar(
        "DEPARTAMENTO",
        "6 departamentos sin tilde (PETEN, QUICHE, SACATEPEQUEZ, SOLOLA, SUCHITEPEQUEZ, TOTONICAPAN).",
        "Se restaura la tilde oficial usando un catalogo de 22 departamentos + el caso especial CIUDAD CAPITAL.",
        n_tilde,
        "DEPARTAMENTAL (misma fuente) ya usa tilde para los mismos lugares; se unifica hacia la ortografia correcta, no hacia la mas frecuente.",
    )

    # --- MUNICIPIO ---------------------------------------------------------
    limpio, valido = Columnas.procesar_municipio(df["DEPARTAMENTO"], df["MUNICIPIO"])
    n_cambios = int((limpio != df["MUNICIPIO"]).sum())
    df["MUNICIPIO"] = limpio
    df["MUNICIPIO_VALIDO"] = valido
    registrar(
        "MUNICIPIO",
        f"Formato/tilde inconsistente en {n_cambios} registros; {int((~valido).sum())} fuera del catalogo oficial.",
        "Se normaliza contra el catalogo de municipios por departamento (fuente: Wikipedia/INE, verificado contra los datos). CIUDAD CAPITAL usa un dominio propio (ZONA 1..25).",
        n_cambios,
        "El catalogo se verifico contra los datos reales: los 'faltantes' son municipios pequenos sin ningun establecimiento de diversificado, no errores de escritura.",
    )

    # --- ESTABLECIMIENTO -----------------------------------------------------
    limpio, mapa_tildes = Columnas.procesar_establecimiento(df["ESTABLECIMIENTO"])
    n_cambios = int((limpio != df["ESTABLECIMIENTO"]).sum())
    df["ESTABLECIMIENTO"] = limpio
    registrar(
        "ESTABLECIMIENTO",
        "70 registros con comillas envolviendo el nombre completo; 446 grupos con variantes que solo difieren en tildes (incluye typos de acento grave).",
        "Se quitan comillas que envuelven TODO el campo (se conservan comillas internas, ej. COLEGIO \"LA PATRIA\"); se unifican variantes acento-insensibles hacia la forma con tilde correcta.",
        n_cambios,
        "No se cambia ninguna letra del nombre ni se corrigen typos reales: eso se deja para revision manual en el listado de duplicados parciales.",
    )

    # --- DIRECCION, SUPERVISOR, DIRECTOR (texto libre + placeholders) ------
    for col in ["DIRECCION", "SUPERVISOR", "DIRECTOR"]:
        limpio, mapa_tildes_col = Columnas.procesar_texto_libre_con_placeholder(df[col])
        registrar(
            col,
            "Vacios y placeholders ('-', '--', '.', 'S/D', 'SIN REGISTRO', etc.) mezclados como si fueran texto valido; "
            f"{len(mapa_tildes_col)} variantes que solo difieren en tildes (ej. nombres de personas escritos con y sin acento).",
            "Se normalizan espacios/caracteres invisibles/puntuacion suelta, se unifican variantes de tilde hacia la forma acentuada, y se convierten los placeholders a NA explicito.",
            int(limpio.isna().sum()),
            "Contar '---' como dato real subestima el porcentaje verdadero de valores faltantes; dos tildes distintas del mismo nombre no son dos personas distintas.",
        )
        df[col] = limpio

    # --- TELEFONO -----------------------------------------------------------
    resultado_tel = procesar_telefono(df["TELEFONO"].tolist())
    n_multiples = int((pd.Series(resultado_tel["TELEFONO_CANTIDAD"]) > 1).sum())
    n_invalidos = int((~pd.Series(resultado_tel["TELEFONO_VALIDO"])).sum())
    df["TELEFONO_ORIGINAL"] = df["TELEFONO"]
    df["TELEFONO"] = resultado_tel["TELEFONO"]
    df["TELEFONO_CANTIDAD"] = resultado_tel["TELEFONO_CANTIDAD"]
    df["TELEFONO_VALIDO"] = resultado_tel["TELEFONO_VALIDO"]
    df["TELEFONO_OBSERVACION"] = resultado_tel["TELEFONO_OBSERVACION"]
    registrar(
        "TELEFONO",
        "946 vacios; multiples numeros en una sola celda con separadores distintos (coma, '/', 'Y', guion); guion ambiguo (formato interno vs. separador vs. sufijo abreviado); presencia de FAX/extension/rangos.",
        "Se separan los numeros en una lista (TELEFONO), y se agregan TELEFONO_CANTIDAD, TELEFONO_VALIDO y TELEFONO_OBSERVACION. Los sufijos abreviados que comparten prefijo (ej. '22202870-73') se reconstruyen; TELEFONO_ORIGINAL conserva el valor crudo para auditoria.",
        n_multiples + n_invalidos,
        "Riesgo: la reconstruccion de sufijos asume que comparten el prefijo del numero anterior; queda expuesta en TELEFONO_ORIGINAL por si se debe revisar.",
    )

    # --- Categoricas de dominio pequeno (ya se verificaron limpias) --------
    for col in COLUMNAS_CATEGORICAS_SIMPLES:
        limpio = Columnas.procesar_categorica_simple(df[col])
        n_cambios = int((limpio != df[col]).sum())
        df[col] = limpio
        registrar(
            col,
            "Ninguno detectado en el perfilado (categorias ya uniformes en mayusculas).",
            "Se normalizan espacios/mayusculas de forma defensiva y se documenta el dominio observado.",
            n_cambios,
            "Se ejecuta igual sobre esta variable para dejar la verificacion explicita en el codigo, no solo asumida.",
        )

    return df


def verificar_consistencia_departamento_departamental(df: pd.DataFrame) -> str:
    """DEPARTAMENTAL reparte 'GUATEMALA' (departamento) en 4 zonas cardinales
    y 'QUICHE' en 2. Se comprueba que esa relacion se cumpla siempre (item
    5.h de la guia: consistencia entre variables)."""
    grupo_guatemala = {"GUATEMALA NORTE", "GUATEMALA OCCIDENTE", "GUATEMALA SUR", "GUATEMALA ORIENTE"}
    filas_guatemala = df[df["DEPARTAMENTO"].isin(["GUATEMALA", "CIUDAD CAPITAL"])]
    ok_guatemala = filas_guatemala["DEPARTAMENTAL"].isin(grupo_guatemala).all()
    otras_filas_con_zona_guatemala = df[~df["DEPARTAMENTO"].isin(["GUATEMALA", "CIUDAD CAPITAL"])]["DEPARTAMENTAL"].isin(grupo_guatemala).any()

    grupo_quiche = {"QUICHÉ", "QUICHÉ NORTE"}
    filas_quiche = df[df["DEPARTAMENTO"] == "QUICHÉ"]
    ok_quiche = filas_quiche["DEPARTAMENTAL"].isin(grupo_quiche).all()
    otras_filas_con_zona_quiche = df[df["DEPARTAMENTO"] != "QUICHÉ"]["DEPARTAMENTAL"].isin(grupo_quiche).any()

    consistente = ok_guatemala and not otras_filas_con_zona_guatemala and ok_quiche and not otras_filas_con_zona_quiche
    return (
        f"DEPARTAMENTO<->DEPARTAMENTAL consistente: {consistente} "
        f"(Guatemala/Ciudad Capital -> 4 zonas: {ok_guatemala}; Quiche -> 2 zonas: {ok_quiche})"
    )


def main() -> None:
    REPORTES_DIR.mkdir(parents=True, exist_ok=True)
    # La funcion puede ser invocada varias veces en una misma sesion de notebook.
    # Evitamos que el registro acumule filas de ejecuciones anteriores.
    registro_transformaciones.clear()

    df_crudo = cargar_crudo()
    faltantes_antes, variables_con_na_antes = contar_faltantes_estilo_crudo(df_crudo)
    duplicados_exactos_antes = int(df_crudo.duplicated(keep=False).sum())
    unicos_establecimiento_antes = df_crudo["ESTABLECIMIENTO"].nunique()
    candidatos_antes = Duplicados.detectar_duplicados_parciales(df_crudo, umbral=90.0)
    parciales_antes = candidatos_antes[candidatos_antes["similitud_nombre"] < 100]

    df_limpio = limpiar(df_crudo)

    print(verificar_consistencia_departamento_departamental(df_limpio))
    n_municipio_invalido = int((~df_limpio["MUNICIPIO_VALIDO"]).sum())
    if n_municipio_invalido:
        print("Municipios fuera de catalogo tras la limpieza:")
        print(
            df_limpio.loc[~df_limpio["MUNICIPIO_VALIDO"], ["CODIGO", "DEPARTAMENTO", "MUNICIPIO"]]
            .drop_duplicates()
        )

    candidatos_despues = Duplicados.detectar_duplicados_parciales(df_limpio, umbral=90.0)
    # Los pares con 100% de similitud de nombre corresponden casi siempre al
    # mismo establecimiento con varios renglones (una fila por jornada/plan,
    # o por cambio de STATUS en el tiempo): se documentan aparte en vez de
    # mezclarlos con los candidatos reales de error ortografico/tipografico.
    parciales_despues = candidatos_despues[candidatos_despues["similitud_nombre"] < 100]
    nombres_repetidos = Duplicados.resumir_nombres_repetidos(df_limpio)
    nombres_repetidos_distinta_direccion = int((nombres_repetidos["direcciones_distintas"] > 1).sum())

    snapshot_antes = {
        "Registros": len(df_crudo),
        "Variables": len(df_crudo.columns),
        "Valores faltantes": faltantes_antes,
        "% faltantes": round(100 * faltantes_antes / (len(df_crudo) * len(COLUMNAS_ORIGINALES)), 2),
        "Variables con NA": variables_con_na_antes,
        "Duplicados exactos (fila completa)": duplicados_exactos_antes,
        "Posibles duplicados parciales (nombre similar, no identico)": len(parciales_antes),
        "Establecimientos unicos (ESTABLECIMIENTO)": unicos_establecimiento_antes,
    }
    snapshot_despues = snapshot_calidad_despues(df_limpio)
    snapshot_despues["Posibles duplicados parciales (nombre similar, no identico)"] = len(parciales_despues)
    snapshot_despues["Establecimientos unicos (ESTABLECIMIENTO)"] = df_limpio["ESTABLECIMIENTO"].nunique()
    snapshot_despues["Duplicados exactos (fila completa)"] = snapshot_despues.pop("Duplicados exactos")

    informe = pd.DataFrame(
        {
            "Metrica": list(snapshot_antes.keys()),
            "Antes": list(snapshot_antes.values()),
            "Despues": [snapshot_despues.get(k, "") for k in snapshot_antes.keys()],
        }
    )

    tabla_transformaciones = pd.DataFrame(registro_transformaciones)

    df_limpio.to_csv(SALIDA_CSV, index=False, encoding="utf-8")
    tabla_transformaciones.to_csv(REPORTES_DIR / "RegistroTransformaciones.csv", index=False, encoding="utf-8")
    informe.to_csv(REPORTES_DIR / "InformeCalidadAntesDespues.csv", index=False, encoding="utf-8")
    parciales_despues.to_csv(REPORTES_DIR / "PosiblesDuplicadosParciales.csv", index=False, encoding="utf-8")
    nombres_repetidos.to_csv(REPORTES_DIR / "EstablecimientosNombreRepetido.csv", index=False, encoding="utf-8")

    print()
    print("=== Informe de calidad: antes vs despues ===")
    print(informe.to_string(index=False))
    print()
    print(f"Dataset limpio: {SALIDA_CSV}")
    print(f"Registro de transformaciones: {REPORTES_DIR / 'RegistroTransformaciones.csv'} ({len(tabla_transformaciones)} filas)")
    print(f"Posibles duplicados parciales (nombre similar, no identico) para revision manual: {REPORTES_DIR / 'PosiblesDuplicadosParciales.csv'} ({len(parciales_despues)} pares)")
    print(
        f"Establecimientos con nombre repetido en el mismo municipio: {REPORTES_DIR / 'EstablecimientosNombreRepetido.csv'} "
        f"({len(nombres_repetidos)} grupos, {nombres_repetidos_distinta_direccion} con direccion distinta -> revisar esos primero)"
    )


if __name__ == "__main__":
    main()


DEPARTAMENTO<->DEPARTAMENTAL consistente: True (Guatemala/Ciudad Capital -> 4 zonas: True; Quiche -> 2 zonas: True)
Municipios fuera de catalogo tras la limpieza:
              CODIGO   DEPARTAMENTO         MUNICIPIO
1262   20-09-0031-46     CHIQUIMULA     QUEZALTEPEQUE
1263   20-09-0037-46     CHIQUIMULA     QUEZALTEPEQUE
1264   20-09-0038-46     CHIQUIMULA     QUEZALTEPEQUE
1265   20-09-0039-46     CHIQUIMULA     QUEZALTEPEQUE
1266   20-09-0040-46     CHIQUIMULA     QUEZALTEPEQUE
1267   20-09-0041-46     CHIQUIMULA     QUEZALTEPEQUE
1268   20-09-0043-46     CHIQUIMULA     QUEZALTEPEQUE
1269   20-09-0044-46     CHIQUIMULA     QUEZALTEPEQUE
1270   20-09-0045-46     CHIQUIMULA     QUEZALTEPEQUE
1271   20-09-0056-46     CHIQUIMULA     QUEZALTEPEQUE
1272   20-09-0058-46     CHIQUIMULA     QUEZALTEPEQUE
1273   20-09-0059-46     CHIQUIMULA     QUEZALTEPEQUE
1274   20-09-0060-46     CHIQUIMULA     QUEZALTEPEQUE
1275   20-09-0063-46     CHIQUIMULA     QUEZALTEPEQUE
1276   20-09-0064-46     CH